In [ ]:
import os,sys,time,hashlib,subprocess,urllib.request
T0=time.monotonic(); LIMIT=30*60
print('START',time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()),'Python',sys.version.split()[0],flush=True)
subprocess.run([sys.executable,'-m','pip','install','-q','fair-esm==2.0.0','pandas==2.2.3'],check=True,timeout=240)
import torch,esm,pandas as pd,numpy as np
print('VERSIONS torch',torch.__version__,'esm',getattr(esm,'__version__','2.0.0'),'pandas',pd.__version__,'available_gb',round(os.sysconf('SC_PAGE_SIZE')*os.sysconf('SC_AVPHYS_PAGES')/1e9,2),flush=True)
assert os.sysconf('SC_PAGE_SIZE')*os.sysconf('SC_PHYS_PAGES') > 8_000_000_000
urls={'RBPbase.csv':'https://zenodo.org/api/records/11061100/files/RBPbase.csv/content','esm2_embeddings_rbp.csv':'https://zenodo.org/api/records/11061100/files/esm2_embeddings_rbp.csv/content'}
for fn,url in urls.items(): urllib.request.urlretrieve(url,'/tmp/'+fn)
assert hashlib.md5(open('/tmp/RBPbase.csv','rb').read()).hexdigest()=='fc235f3acfcd4dcadb71527bdf6704a3'
assert hashlib.sha256(open('/tmp/esm2_embeddings_rbp.csv','rb').read()).hexdigest()=='c282a434f92b6702d54baa821adcdb563070020e9b9fe29a045a1515e53bc990'
ids=['K17alfa62_gp87','K65PH164_gp148','K50PH164C1_gp20']
r=pd.read_csv('/tmp/RBPbase.csv'); ref=pd.read_csv('/tmp/esm2_embeddings_rbp.csv')
assert len(r)==274 and len(ref)==274 and all(r.protein_ID.value_counts().get(i)==1 for i in ids)
print('MODEL', 'esm2_t33_650M_UR50D','ids',ids,'tolerance max_abs <= 1e-4 AND cosine >= 0.999999; abort on mismatch',flush=True)
torch.set_num_threads(2)
model,alphabet=esm.pretrained.esm2_t33_650M_UR50D();model.eval();conv=alphabet.get_batch_converter()
for k in ['esm2_t33_650M_UR50D.pt','esm2_t33_650M_UR50D-contact-regression.pt']:
 p=os.path.expanduser('~/.cache/torch/hub/checkpoints/'+k)
 if os.path.isfile(p): print('WEIGHTS',k,os.path.getsize(p),hashlib.sha256(open(p,'rb').read()).hexdigest(),flush=True)
for pid in ids:
 assert time.monotonic()-T0 < LIMIT
 row=r[r.protein_ID==pid].iloc[0]; seq=row.protein_sequence
 _,_,tokens=conv([(pid,seq)])
 with torch.no_grad(): out=model(tokens,repr_layers=[33],return_contacts=True)
 vec=out['representations'][33][0,1:len(seq)+1].mean(0).cpu().numpy()
 reference=ref[ref.protein_ID==pid].iloc[0,2:].to_numpy(dtype=float)
 err=np.max(np.abs(vec-reference));cos=np.dot(vec,reference)/(np.linalg.norm(vec)*np.linalg.norm(reference))
 print('RESULT',pid,'len',len(seq),'max_abs',float(err),'cosine',float(cos),'PASS',bool(err<=1e-4 and cos>=.999999),flush=True)
 del out,tokens
print('ELAPSED_SEC',round(time.monotonic()-T0,1),flush=True)
